# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
result = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1")
print(result)
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
print(month_path)

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content (content_hash_id), on one day (report_date),
for one client (client_hash_id). This is a daily fact table. I'll use the
mid-panel month 2026-03 to build and test my contract, avoiding the final
sealed month (June 2026) for any label logic, per the warehouse warning.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{month_path}')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate (content, date) rows found:", grain_check.shape[0])
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content, date) rows found: 0


,content_hash_id,report_date,n


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields: gsc_impressions, gsc_clicks, gsc_sum_position, scroll_events,
ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta /
ai_other (AI-referral session counts), sessions_ai.

Label/proxy field: a decline signal derived from gsc_clicks and gsc_impressions
trends over time (built in later modeling weeks, not here).

Context fields: client_hash_id, content_hash_id, report_date, month.

Excluded: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
— these describe whether data exists, not content performance. Including them
as features would let a model prefer well-instrumented clients over genuinely
well-performing content, so I exclude them deliberately.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# qualitative field sorting — no computation needed here


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

March 2026: 9,841,378 rows total. GSC data available on 3,611,061 rows
(about 37%), GA4 data available on only 413,966 rows (about 4%) — GA4
coverage is far sparser than GSC, an important limitation for any feature
relying on engagement data like scroll_events or sessions_ai.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{month_path}')
""").df()
counts

# Section 3 — code cell (Query 3: availability with IS TRUE)

availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{month_path}')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


**Section 3 continued — 5 features (add a new markdown + code cell here)**

Five features built from the March 2026 slice, each with an "available at
decision moment" justification:

1. gsc_impressions — available same-day, logged by Search Console daily sync.
2. gsc_clicks — available same-day, same daily sync as impressions.
3. gsc_sum_position — available same-day, part of the same GSC row.
4. scroll_events — available same-day, captured by page-level analytics tracking.
5. sessions_ai — available same-day, aggregated AI-referral session count logged
   in the same daily fact row.

All five are same-day observed metrics, not future-looking or label-derived,
so they're safe to use as predictors at the moment a scoring decision is made.


In [16]:
features = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_sum_position,
           scroll_events, sessions_ai
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 20000
""").df()
print(features.shape)
features.head()

(20000, 8)


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,67,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,616,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,28,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,25,<NA>,<NA>


**The leakage trap (new markdown + code cell, still in Section 3)**

Leakage trap: I'll add one label-derived column on purpose — a "future_clicks"
value that's actually just gsc_clicks shifted forward — compute a quick score
predicting decline using it, watch the score jump toward suspiciously perfect,
then delete the column and report the honest score instead.

In [17]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Build a fake "proxy target": next period clicks (this is realistic and fine)
X = features[["gsc_impressions", "gsc_sum_position", "scroll_events", "sessions_ai"]].fillna(0)
y = features["gsc_clicks"].fillna(0)

# HONEST baseline model (no leakage)
model_honest = LinearRegression().fit(X, y)
score_honest = r2_score(y, model_honest.predict(X))
print("Honest R2:", score_honest)

# LEAKY version: sneak the label itself in as a "feature"
X_leaky = X.copy()
X_leaky["leaked_clicks"] = y  # the trap: this IS the target
model_leaky = LinearRegression().fit(X_leaky, y)
score_leaky = r2_score(y, model_leaky.predict(X_leaky))
print("Leaky R2 (should jump toward 1.0):", score_leaky)

# Remove the leak, keep the honest number
del X_leaky["leaked_clicks"]
print("Final honest score kept:", score_honest)

Honest R2: 0.5256907683372593
Leaky R2 (should jump toward 1.0): 1.0
Final honest score kept: 0.5256907683372593


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you: it's an unbalanced panel — clients have
different GSC/GA4 history start dates (dim_clients.gsc_data_start /
ga4_data_start), so early rows for some clients may simply not exist rather
than reflect zero performance. Early-period rows can also be GSC-only if GA4
wasn't yet connected for that client, understating engagement metrics like
scroll_events for those rows. One named limitation: any client with
gsc_data_available = FALSE for March 2026 has no real signal that month —
those rows should not be treated as "zero traffic," just "not observed."

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

missing_check = con.sql(f"""
    SELECT COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_missing_rows
    FROM read_parquet('{month_path}')
""").df()
missing_check


,gsc_missing_rows
0,6230317


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.